# Гибридная рекомендательная система BookShelf

Продолжение `EDA_AmazonBooks.ipynb`: EDA не повторяется. Данные берутся только из `data/AmazonBooks/`. Сравниваем популярность, мультиязычные эмбеддинги, item-based adjusted cosine и гибрид. Общее ядро в `src/recsys/engine.py` используется здесь и в отдельном gRPC-процессе.

Установка из корня: `python -m pip install -e ".[dev,recommendations]"`. Первый запуск скачивает веса в `.cache/models/`. Датасеты не скачиваются; модель не дообучаем.

> Исторический прогон первоначальной версии ядра. После перехода на Goodbooks исправлено нормирование CF и popularity разделена на частотную популярность и bayesian. Сохранённые выводы ниже не пересчитывались; повторное выполнение использует обновлённое ядро и даст другие результаты. Актуальное сравнение находится в `Goodbooks_Recommendations.ipynb`.

In [1]:
import gc
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "data/AmazonBooks/Books_rating.csv").is_file(), "Open notebook from project root"
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

import numpy as np
import pandas as pd
from IPython.display import display
from ml.datasets import load_amazon, temporal_split, prepare_catalog, encode_ratings
from ml.run_eval import evaluate
from recsys.embeddings import FastEmbedEncoder, DEFAULT_MODEL
from recsys.engine import HybridRecommender
from recsys.domain import UserProfile

# 0 означает весь локальный файл; для быстрого прогона BOOKSHELF_MAX_RATINGS=300000.
MAX_ROWS = int(os.environ.get("BOOKSHELF_MAX_RATINGS", "0")) or None
MAX_BOOKS = int(os.environ.get("BOOKSHELF_MAX_BOOKS", "1500"))
MAX_USERS = int(os.environ.get("BOOKSHELF_EVAL_USERS", "100"))
print({"max_rows": MAX_ROWS, "max_books": MAX_BOOKS, "eval_users": MAX_USERS, "model": DEFAULT_MODEL})

{'max_rows': None, 'max_books': 1500, 'eval_users': 100, 'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}


## Подготовка и временное разбиение

Сохраняем Amazon `Id` как ID издания; название используется только для присоединения описания и жанров, как в существующем EDA. При нескольких строках метаданных выбирается первая: неоднозначность остаётся ограничением датасета. Оценки 1–5 переводим в 2–10, production принимает всю шкалу 1–10.

Границы 80/90% определяются по времени событий, одинаковое время не пересекает границу. Последнюю оценку пары выбираем **внутри каждого периода**: будущая оценка не удаляет прошлую. Частоты и каталог определяем только по train. Достоверного времени публикации нет, поэтому кандидаты - издания, уже наблюдавшиеся в train.

По умолчанию используются первые 300 000 строк CSV, не случайная репрезентативная выборка. Для всего датасета: `MAX_ROWS=None` или переменная `BOOKSHELF_MAX_RATINGS=0`.

Оба локальных CSV проверены целиком: кодировка UTF-8. Новый загрузчик использует UTF-8, чтобы сохранить неанглийские символы; исходный EDA не изменяется.

In [2]:
data = load_amazon(ROOT / "data", max_rows=MAX_ROWS)
train, validation, test = temporal_split(data)
assert train["review/time"].max() < validation["review/time"].min()
assert validation["review/time"].max() < test["review/time"].min()
books, book_ids, train = prepare_catalog(ROOT / "data", train, max_books=MAX_BOOKS)
user_ids = {uid: i + 1 for i, uid in enumerate(sorted(data.loc[data["Id"].isin(book_ids), "User_id"].unique()))}
train_ratings = encode_ratings(train, book_ids, user_ids)
display(pd.DataFrame([
    {"split": name, "ratings": len(part), "users": part.User_id.nunique(),
     "start": pd.to_datetime(part["review/time"].min(), unit="s", utc=True),
     "end": pd.to_datetime(part["review/time"].max(), unit="s", utc=True)}
    for name, part in [("train (selected catalog)", train), ("validation", validation), ("test", test)]
]))
print("Books:", len(books), "Training ratings:", len(train_ratings))
# Полный исходный DataFrame и ненужные текстовые поля больше не удерживаем.
del data
for part in (train, validation, test):
    part.drop(columns=["Title", "review/score"], inplace=True)
    for col in ("Id", "User_id"):
        part[col] = part[col].cat.remove_unused_categories()
gc.collect()


,split,ratings,users,start,end
0,train (selected catalog),571824,204234,1996-12-25 00:00:00+00:00,2011-10-30 00:00:00+00:00
1,validation,242486,118647,2011-10-31 00:00:00+00:00,2012-11-21 00:00:00+00:00
2,test,243110,109466,2012-11-22 00:00:00+00:00,2013-03-04 00:00:00+00:00


Books: 1500 Training ratings: 571824


34

## Мультиязычные эмбеддинги и гибрид

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` через FastEmbed/ONNX, CPU. Кэш учитывает имя модели и весь текст. Название и жанры идут перед описанием, чтобы сохраниться при усечении входа моделью.

Content: профиль из векторов книг с оценкой ≥8, взвешенных `(rating-5)/5`, плюс вектор отмеченных жанров. Близость - cosine, отрицательные значения обрезаются до нуля.

CF: центрируем оценки по среднему каждого читателя. Для пары книг числитель и оба слагаемых знаменателя adjusted cosine считаются **по общим читателям**. Требуем ≥2 общих читателей и ≥3 оценки кандидата, используем shrinkage `n_common/(n_common+10)`. Положительное сходство взвешиваем предпочтением `(rating-5.5)/4.5`: низкая оценка уменьшает балл похожих книг. Нулевая дисперсия даёт ноль, а не NaN.

`score=(1-w)*content+w*CF`, где `w=cf_weight*n_ratings/(n_ratings+10)`. Без CF-свидетельств `w=0`. Без профиля и CF используем байесовскую популярность. Оценённые, прочитанные и неопубликованные книги исключаются. До 10 рекомендаций при достаточном числе кандидатов.

In [3]:
encoder = FastEmbedEncoder(cache_dir=str(ROOT / ".cache/models"))
model = HybridRecommender(encoder)
train_catalog = model.prepare(books, train_ratings)
print("Embeddings:", train_catalog.vectors.shape, "Sparse ratings:", train_catalog.centered.shape, "nnz:", train_catalog.centered.nnz)
assert np.isfinite(train_catalog.vectors).all()

Embeddings: (1500, 384) Sparse ratings: (1500, 204234) nnz: 571824


## Validation: сравнение и выбор веса

Положительные события - оценки ≥8. Ground truth исключает известные пользователю книги и издания вне фиксированного каталога. Ранжируем весь оставшийся каталог без sampled negatives. Метрики усредняются по пользователям с доступной релевантной книгой. Детерминированная выборка пользователей: seed=42. Показываем долю holdout в каталоге и число новых пользователей.

Вес выбираем только на validation по NDCG@10. Выигрыш гибрида заранее не предполагается.

Отдельно сравниваем cohort=all и cohort=warm. Вес выбираем по warm-validation, чтобы измерять персонализацию; у новых пользователей без заданных интересов закономерно работает популярность. Размер популяции и размер выборки указаны отдельно.

In [4]:
validation_rows = []
for mode, weight in [("popularity", None), ("content", None), ("cf", None),
                     ("hybrid", 0.25), ("hybrid", 0.5), ("hybrid", 0.75)]:
    if weight is not None:
        model.cf_weight = weight
    for cohort in ["all", "warm"]:
        validation_rows.append({"model": mode, "cohort": cohort, "cf_weight": weight,
                                **evaluate(model, train_catalog, validation, book_ids, user_ids, mode=mode,
                                           cohort=cohort, max_users=MAX_USERS)})
validation_results = pd.DataFrame(validation_rows)
display(validation_results)
selection = validation_results[validation_results.model.eq("hybrid") & validation_results.cohort.eq("warm")]
assert selection.users.max() > 0, "No warm validation users: increase the dataset size"
best = selection.sort_values("ndcg@10", ascending=False, kind="stable").iloc[0]
best_weight = float(best.cf_weight)
print("Selected CF weight (warm validation):", best_weight)

,model,cohort,cf_weight,recall@10,ndcg@10,coverage@10,users,eligible_users,cold_users,holdout_catalog_fraction
0,popularity,all,NaN,0.000000,0.000000,0.006667,100,23029,91,0.350833
1,popularity,warm,NaN,0.021667,0.019335,0.007333,100,1807,0,0.350833
2,content,all,NaN,0.000000,0.000000,0.050667,100,23029,91,0.350833
3,content,warm,NaN,0.027076,0.016674,0.294667,100,1807,0,0.350833
4,cf,all,NaN,0.000000,0.000000,0.037333,100,23029,91,0.350833
5,cf,warm,NaN,0.012909,0.011657,0.216667,100,1807,0,0.350833
6,hybrid,all,0.25,0.000000,0.000000,0.053333,100,23029,91,0.350833
7,hybrid,warm,0.25,0.024613,0.019070,0.318000,100,1807,0,0.350833
8,hybrid,all,0.50,0.003333,0.001672,0.057333,100,23029,91,0.350833
9,hybrid,warm,0.50,0.014613,0.014108,0.322000,100,1807,0,0.350833


Selected CF weight (warm validation): 0.25


## Test: фиксируем параметры

Объединяем train и validation, оставляя последнюю известную оценку каждой пары. Каталог и выбранный вес фиксированы. Test не используется для выбора параметров. Это batch-оценка: внутри test история не обновляется.

In [5]:
# Освобождаем подготовленный train-каталог перед расчётом по train+validation.
del train_catalog, train_ratings
gc.collect()
fit_data = pd.concat([train, validation[validation.Id.isin(book_ids)]], ignore_index=True)
fit_data = fit_data.sort_values("review/time", kind="stable").drop_duplicates(["User_id", "Id"], keep="last")
model.cf_weight = best_weight
test_catalog = model.prepare(books, encode_ratings(fit_data, book_ids, user_ids))
test_results = pd.DataFrame([
    {"model": mode, "cohort": cohort, **evaluate(model, test_catalog, test, book_ids, user_ids, mode=mode, cohort=cohort, max_users=MAX_USERS)}
    for mode in ["popularity", "content", "cf", "hybrid"]
    for cohort in ["all", "warm", "cold"]
])
display(test_results)
artifacts = ROOT / "notebooks/artifacts"
artifacts.mkdir(parents=True, exist_ok=True)
validation_results.to_csv(artifacts / "validation.csv", index=False)
test_results.to_csv(artifacts / "test.csv", index=False)

,model,cohort,recall@10,ndcg@10,coverage@10,users,eligible_users,cold_users,holdout_catalog_fraction
0,popularity,all,0.010,0.006309,0.006667,100,26402,97,0.448056
1,popularity,warm,0.005,0.001772,0.007333,100,1224,0,0.448056
2,popularity,cold,0.010,0.010000,0.006667,100,25178,100,0.448056
3,content,all,0.010,0.006309,0.020000,100,26402,97,0.448056
4,content,warm,0.060,0.024577,0.327333,100,1224,0,0.448056
5,content,cold,0.010,0.010000,0.006667,100,25178,100,0.448056
6,cf,all,0.010,0.006309,0.026000,100,26402,97,0.448056
7,cf,warm,0.025,0.009428,0.228667,100,1224,0,0.448056
8,cf,cold,0.010,0.010000,0.006667,100,25178,100,0.448056
9,hybrid,all,0.010,0.006309,0.026000,100,26402,97,0.448056


## Примеры и холодный старт

In [6]:
uid = max(test_catalog.user_ratings, key=lambda u: sum(s >= 8 for s in test_catalog.user_ratings[u].values()))
history = test_catalog.user_ratings[uid]
display(pd.DataFrame([{"title": books[test_catalog.index[bid]].title, "rating": score} for bid, score in history.items()]))
display(pd.DataFrame([vars(r) for r in model.rank(test_catalog, UserProfile(uid))]))
print("New reader with genre interests:")
display(pd.DataFrame([vars(r) for r in model.rank(test_catalog, UserProfile(-1, interests=("science fiction",)))]))
print("New reader without history:")
display(pd.DataFrame([vars(r) for r in model.rank(test_catalog, UserProfile(-1))]))
assert not ({r.book_id for r in model.rank(test_catalog, UserProfile(uid))} & set(history))

,title,rating
0,1984,10.0
1,Nineteen Eighty-four,10.0
2,George Orwell 1984,10.0
3,1984,10.0
4,Eaters Of The Dead,8.0
...,...,...
316,Treasure Island; (Easy reading books: upper-gr...,8.0
317,Treasure Island,8.0
318,everything on this page is for Treasure Island,8.0
319,Treasure Island (Classic Illustrated),8.0


,book_id,title,score
0,1491,Middlesex [Unabridged Audiobook],0.803449
1,1163,The Color of Water,0.798066
2,282,Case Histories,0.784291
3,551,The Summons,0.783083
4,715,Lord of the flies: A novel,0.779475
5,1439,OF MICE AND MEN,0.775515
6,1316,Of Mice And Men,0.771080
7,1036,Of Mice and Men,0.770484
8,1184,Of Mice and Men,0.770484
9,1005,Of Mice and Men,0.770484


New reader with genre interests:


,book_id,title,score
0,1008,the illustrated man,0.702778
1,947,The Illustrated Man,0.684856
2,1420,The Illustrated Man,0.684856
3,158,Everything's Eventual: 14 Dark Tales,0.674465
4,938,Hell House,0.653960
5,753,The illustrated man (Corgi science fiction),0.645669
6,541,Point of origin,0.636585
7,770,The call of the wild,0.628029
8,1163,The Color of Water,0.627961
9,560,Shadow of the Hegemon,0.622425


New reader without history:


,book_id,title,score
0,898,With the Old Breed: At Peleliu and Okinawa,0.980021
1,331,Tear Soup: A Recipe for Healing After Loss,0.974506
2,657,Cooky book,0.972759
3,705,Betty Crocker's cooky book,0.972759
4,1458,Betty Crockers Cooky Book (1st Edition),0.972759
5,481,Harrington on Hold 'em Expert Strategy for No ...,0.969224
6,11,Where the Sidewalk Ends: Poems and Drawings,0.968980
7,935,CARS AND TRUCKS AND THINGS THAT GO,0.968394
8,351,Left to Tell: Discovering God Amidst The Rwand...,0.963586
9,709,The monster at the end of this book (A little ...,0.963295


## Интерпретация и перенос в ЛР3

Таблицы содержат фактические метрики для выбранных ограничений строк, каталога и пользователей. Для выводов о всём Amazon нужен повтор с `MAX_ROWS=None` и увеличенным каталогом. Первые строки CSV не являются репрезентативной выборкой. Сравнивайте гибрид и с content, и с популярностью; слабый CF при разреженных данных не оправдывает использование test при настройке.

Ограничения: усечение описаний моделью, неоднозначность метаданных по названию, отсутствие достоверного времени публикации. Offline-метрики не доказывают рост читательской активности. Десять рекомендаций возможны при наличии десяти допустимых кандидатов.

API передаёт актуальные книги, оценки, интересы и исключения через Protocol Buffers, отдельный процесс вызывает это же ядро. Amazon ID не смешиваются с ID production-БД: HTTP-рекомендации строятся по каталогу BookShelf.

Источники: [модель](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), [FastEmbed](https://qdrant.github.io/fastembed/examples/Supported_Models/).

**Сохранённый полный прогон (UTF-8):** 571 824 обучающих оценки, 1 500 книг; выбран `cf_weight=0.25`. На 100 тестовых читателях с историей content: Recall@10=0.0600, NDCG@10=0.0246, coverage=0.3273; гибрид: 0.0500, 0.0245, 0.3467. Таким образом, content точнее на этой выборке, а гибрид покрывает больше книг. Тестовые результаты не использовались для повторного подбора веса. Из 1 224 доступных warm-пользователей оценивается выборка из 100; доверительные интервалы здесь не рассчитывались.